# X-Mobility 파인튜닝 — Kaggle (E · 서진석)

`e2e/finetune.py` 를 Kaggle 무료 GPU(T4 × 2)에서 돌린다. 데이터 = 교사 데이터 xm_v1(MANIFEST `xm_parking_r3` v1.0), 시작 가중치 = `nvidia/X-Mobility` 사전학습 ckpt.

**한 번만 준비**
1. Kaggle 계정 휴대폰 인증(인증해야 GPU · 인터넷 켜짐).
2. **데이터셋 만들기**: Datasets → New Dataset → `02_xm_v1.zip` 업로드(이름 예: `fd-xm-v1`, Private). Kaggle 이 zip 을 풀어 둠.
3. **HF 토큰**: huggingface.co 에 로그인한 채로 https://huggingface.co/nvidia/X-Mobility 에서 **접근 요청(Agree and access)** — 안 하면 403 "not in the authorized list" → Settings → Access Tokens(Read) → 이 노트북 Add-ons → Secrets 에 이름 `HF_TOKEN` 으로 저장, 노트북에 연결(체크).
4. 이 노트북 오른쪽 Settings: **Accelerator = GPU T4 x2 · Internet = On**, Input 에 위 데이터셋 추가.

**돌리기**
- 처음: 아래 `MODE = "smoke"` 로 Run All(약 30~40분) — 실행기 · 데이터 · ckpt 확인 + 스텝 시간 실측 + **제로샷 오프라인 평가**(사전학습 ckpt 그대로 test split, 결과보고서 표 E-15 제로샷 칸). 마지막 두 칸 출력을 복사해 둘 것(대화형 세션 결과는 저장 안 됨).
- 그다음: `MODE = "full"` 로 바꾸고 **Save Version → Save & Run All (Commit)** — 브라우저를 닫아도 최대 12시간 돈다. 시간 예산에 맞춰 에폭 수를 자동으로 정함.
- 실험 바꾸기: `RUN_NAME` + `EXTRA_ARGS`(예: 명령 가중치 30 = `--gin XMobilityLoss.action_weight=30`, 시드 = `--seed 1`) + 오른쪽 Input 데이터셋. `SAVE_TOP_K = 3` 이면 val 상위 3개 ckpt 를 남겨 에폭 평균(`e2e/avg_ckpt.py`)에 씀.
- 여러 ckpt 재평가 · 체크포인트 평균: `e2e/kaggle_eval.ipynb`(학습 없음 · HF 토큰 필요 없음).
- **CLI(`kaggle kernels push`)로 올린 버전은 Kaggle 비밀(HF_TOKEN)을 못 읽는다**(11.7 확인, 비밀 서비스 HTTP 400). 브라우저에서 Save & Run All 하거나, 사전학습 ckpt(`x_mobility-nav2-semantic_action_path.ckpt`, sha12 339c68f9e027)가 든 비공개 데이터셋 `henryseo711/fd-xm-pretrained` 를 Input 에 붙이면 토큰 없이 돈다(11.7 만듦). 다시 만들 일이 있으면 `SAVE_PRETRAINED = True` 로 한 번 돌린 출력의 `pretrained/` 에서 얻는다.
- 평가만 다시: `MODE = "eval"` — 학습 버전의 Output 을 이 노트북 Input 으로 붙이고 `EVAL_CKPT` 에 그 ckpt 경로(마지막 칸이 후보를 출력함). 10~20분.
- 결과: 버전의 Output 탭에서 `run_meta.json` · `checkpoints/` · `eval_test/summary.json`(파인튜닝 오프라인 평가) · `curves.png` · `curves.md`(학습 곡선) 다운로드 → MANIFEST `checkpoint` 행.

In [ ]:
MODE = "smoke"            # "smoke" → 40배치 시험 + 제로샷 평가 · "full" → 본 학습(+ 평가) · "eval" → 평가만
EVAL_CKPT = ""            # eval: 평가할 ckpt 경로(빈 칸 = 사전학습 ckpt = 제로샷). 학습 버전 Output 을 Input 으로 붙이면 /kaggle/input/... 아래
EVAL_SPLITS = ["test"]    # eval: 평가할 split (val 도 보려면 ["test", "val"])
TIME_BUDGET_H = 10.0      # full: 학습에 쓸 시간(Kaggle 세션 12시간 - 준비 · smoke · test · 학습 뒤 오프라인 평가 여유)
BATCH, ACCUM = 2, 2       # T4 15 GB: 배치 4 는 첫 스텝 OOM(11.6) → 배치 2 × 누적 2 = 실효 배치 그대로(GPU 2장 × 4)
RUN_NAME = "xm_ft_v0.1_p3"  # full 출력 폴더 이름 = MANIFEST checkpoint 버전. 지금까지 v0.1(xm_v1) · v0.2(xm_v1.2 = v1.1 + Nav2 train ×3)
                          # · v0.1_aw30(xm_v1 + 명령 가중치 30) · v0.1_p3(xm_v1 경로 정답 ×1/3 = dataset-04-xm-v1-p3). Input 데이터셋을 이름과 맞출 것
EXTRA_ARGS = ""           # finetune.py 에 덧붙일 인자. 예: aw30 = "--gin XMobilityLoss.action_weight=30" · 시드 분산 = "--seed 1". 에폭 수는 MAX_EPOCHS 로(여기 --gin NUM_EPOCHS 는 cell 8 이 뒤에서 덮어씀)
SAVE_TOP_K = 3            # val 상위 k 개 ckpt(+ last) 저장 — 에폭 7–9 평균(avg_ckpt.py)용. 디스크가 모자라면 1
SAVE_PRETRAINED = False   # True: HF 에서 받은 사전학습 ckpt 를 출력 pretrained/ 에 남김 (fd-xm-pretrained 데이터셋을 만들 때만 — 11.7 에 만듦)
MAX_EPOCHS = 10           # full: 시간이 남아도 이 이상은 안 돌림(gin 기본값)
REPO_URL = "https://github.com/JP-Avengers/FUTURE-DRIVE-PROJECT.git"
XM_URL, XM_REV = "https://github.com/NVlabs/X-Mobility.git", "5837aea"
CKPT_REPO, CKPT_FILE = "nvidia/X-Mobility", "x_mobility-nav2-semantic_action_path.ckpt"
WORK = "/kaggle/working"

In [ ]:
import os, subprocess, glob, json, math, shutil, time
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise SystemExit(f"실패({r.returncode}): {cmd}")
    return r.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
os.chdir("/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp")
if not os.path.isdir("FUTURE-DRIVE-PROJECT"):
    sh(f"git clone --depth 20 {REPO_URL}")
if not os.path.isdir("X-Mobility"):
    sh(f"git clone {XM_URL} && cd X-Mobility && git checkout -q {XM_REV}")
REPO, XM = os.path.abspath("FUTURE-DRIVE-PROJECT"), os.path.abspath("X-Mobility")
sh(f"git -C {REPO} log --oneline -1 && git -C {XM} log --oneline -1")
assert "GIN_SKIP" in open(f"{REPO}/e2e/finetune.py").read(), "레포 main 에 finetune.py 새 판(gin 파싱 수정)이 아직 없음 — push 먼저"

In [ ]:
os.environ["USE_TF"] = "0"  # wandb 0.19.4 가 protobuf 를 5.29 로 내려 Kaggle 의 TF(≥6.31 필요)를 불러오면 죽음 → transformers 가 TF 를 안 보게
# X-Mobility 가 맞춘 판(requirements.txt)과 같게. torch 는 Kaggle 기본(CUDA) 그대로.
# huggingface_hub 는 transformers 4.48.1 범위(<1.0) 안에서 0.25 로 고정. wandb 는 X-Mobility trainer 가 import.
sh("pip install -q gin-config==0.5.0 einops==0.7.0 pytorch-lightning==2.5.0.post0 transformers==4.48.1 "
   "diffusers==0.29.2 huggingface_hub==0.25.2 timm==1.0.14 torcheval==0.0.7 torchmetrics wandb==0.19.4")
sh("python -c \"import torch, transformers, diffusers, pytorch_lightning as pl; "
   "from diffusers import UNet2DConditionModel; from transformers import CLIPTextModel; "
   "print(torch.__version__, torch.cuda.device_count(), transformers.__version__, diffusers.__version__, pl.__version__)\"")

In [ ]:
# 데이터 루트 = train/val/test 가 있는 폴더
cand = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), "val")) and os.path.isdir(os.path.join(os.path.dirname(p), "test"))]
assert cand, "입력 데이터셋에서 train/val/test 를 못 찾음 — 오른쪽 Input 에 xm_v1 데이터셋 추가"
DATA = sorted(cand, key=len)[0]
print("DATA =", DATA)
sh(f"python {REPO}/e2e/check_xm_dataset.py {DATA} --image-stride 20")

In [ ]:
# 사전학습 ckpt — Input 데이터셋(fd-xm-pretrained)에 있으면 그것(HF 토큰 필요 없음 · CLI push 로도 돎), 없으면 HF 에서(비밀 HF_TOKEN 필요 → 브라우저에서 실행)
pre = sorted(glob.glob(f"/kaggle/input/**/{CKPT_FILE}", recursive=True))
if pre:
    CKPT = pre[0]
else:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download
    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        raise SystemExit("HF_TOKEN 을 못 읽음 — CLI 로 올린 버전은 비밀을 못 읽으므로 브라우저에서 Save & Run All, "
                         f"또는 사전학습 ckpt 데이터셋을 Input 에 붙일 것(Add-ons → Secrets 에 HF_TOKEN 도 확인): {e}")
    CKPT = hf_hub_download(CKPT_REPO, CKPT_FILE, token=token, local_dir=os.path.abspath("ckpt"))
    if SAVE_PRETRAINED:
        os.makedirs(f"{WORK}/pretrained", exist_ok=True)
        shutil.copy(CKPT, f"{WORK}/pretrained/{CKPT_FILE}")
print(CKPT, round(os.path.getsize(CKPT) / 2**30, 2), "GB")


In [ ]:
BASE = (f"python {REPO}/e2e/finetune.py --xm-repo {XM} -c {REPO}/e2e/configs/finetune_parking_r3.gin "
        f"-d {DATA} -p {CKPT} --wandb offline --save-top-k {SAVE_TOP_K} {EXTRA_ARGS} "
        f"--gin XMobilityIsaacSimDataModule.batch_size={BATCH} --accumulate-grad-batches {ACCUM}")
smoke = f"{WORK}/smoke"
if MODE in ("smoke", "full") and (not os.path.exists(f"{smoke}/run_meta.json") or "step_time_s" not in json.load(open(f"{smoke}/run_meta.json"))):
    shutil.rmtree(smoke, ignore_errors=True)
    t0 = time.time()
    sh(f"{BASE} -o {smoke} --limit-train-batches 40 --limit-val-batches 4 --no-test --gin NUM_EPOCHS=1")
    print(f"시험 전체 {time.time() - t0:.0f} s")
m = json.load(open(f"{smoke}/run_meta.json")) if os.path.exists(f"{smoke}/run_meta.json") else {"keys": {}}
k = m["keys"] or {}
print(json.dumps({x: m.get(x) for x in ("step_time_s", "steps_per_epoch", "world_size", "batch_size", "effective_batch", "peak_gpu_mem_gb", "global_step")}, indent=1))
print("missing", len(k.get("missing", [])), "· 예상 밖 unexpected", len(k.get("unexpected_other", [])), "· ckpt hparams", k.get("ckpt_hparams"))
assert not k.get("missing"), "missing 키 있음 — 멈춤"

In [ ]:
# 제로샷 오프라인 평가 — smoke 에서만(커밋 실행 시간 아끼려고). 배포 노드와 같게 run 마다 상태 초기화 · 직전 예측 명령 입력
EVAL_BASE = f"python {REPO}/e2e/eval_offline.py --xm-repo {XM} -c {REPO}/e2e/configs/finetune_parking_r3.gin -d {DATA}"
EVAL = f"{EVAL_BASE} --split test"
def show_eval(d):
    if os.path.exists(f"{d}/summary.json"):
        print(json.dumps(json.load(open(f"{d}/summary.json")), indent=1, ensure_ascii=False)[:6000])
if MODE in ("smoke", "full"):   # full 도 — split 이 바뀌면(v1.1 등) 같은 test 로 제로샷을 다시 재야 공정
    zs = f"{WORK}/eval_zeroshot_test"
    if os.path.exists(f"{DATA}/PATH_SCALE.json"):
        print("경로 축척 데이터 — 제로샷 모델은 축척을 안 배워 ADE · FDE 가 틀어지므로 건너뜀(명령 지표는 원래 축척 run 의 제로샷과 같음)")
    elif not os.path.exists(f"{zs}/summary.json"):
        sh(f"{EVAL} -p {CKPT} -o {zs}", check=False)
    show_eval(zs)

In [ ]:
# full: 실측 스텝 시간으로 에폭 수를 정함(검증 · 저장 여유 15 %). lr 스케줄(OneCycle)이 이 에폭 수에 맞춰 끝까지 내려감.
if MODE == "full":
    per_epoch_s = m["steps_per_epoch"] * m["step_time_s"] * 1.15
    epochs = max(1, min(MAX_EPOCHS, int((TIME_BUDGET_H * 3600 - 1200) // per_epoch_s)))
    print(f"에폭당 약 {per_epoch_s / 60:.0f}분 → {epochs} 에폭")
    out = f"{WORK}/{RUN_NAME}_e{epochs}"
    hh = int(TIME_BUDGET_H); mm = int((TIME_BUDGET_H - hh) * 60)
    sh(f'{BASE} -o {out} --gin "NUM_EPOCHS={epochs}" --max-time "00:{hh:02d}:{mm:02d}:00"', check=False)
    rm = json.load(open(f"{out}/run_meta.json"))
    print(json.dumps({k: rm.get(k) for k in ("best_model_path", "monitor", "best_score", "val_loss", "global_step",
                                             "step_time_s", "peak_gpu_mem_gb", "test")}, indent=1, ensure_ascii=False, default=str))
    sh(f"python {REPO}/e2e/plot_training.py {out}", check=False)
    best = rm.get("best_model_path") or f"{out}/checkpoints/last.ckpt"
    sh(f"{EVAL} -p {best} -o {out}/eval_test", check=False)   # 파인튜닝 오프라인 평가(표 E-15 FT 칸)
    show_eval(f"{out}/eval_test")
    zs_dir = f"{WORK}/eval_zeroshot_test"   # 축척 데이터면 제로샷이 없음(cell 7) → --zs 생략
    sh(f"python {REPO}/e2e/fill_report.py --train {out} --ft {out}/eval_test --version {RUN_NAME} "
       + (f"--zs {zs_dir} " if os.path.exists(f"{zs_dir}/summary.json") else "") + f"--out {out}/report_tables.md", check=False)
elif MODE == "eval":
    cands = sorted(glob.glob("/kaggle/input/**/*.ckpt", recursive=True))
    print("ckpt 후보:", *cands, sep="\n  ")
    ck = EVAL_CKPT or CKPT
    # 태그에 run 폴더 이름도 넣음 — v0.1 · aw30 · seed1 의 best 가 모두 epoch=8-step=10719.ckpt 라 파일 이름만으로는 겹침
    tag = "zeroshot" if not EVAL_CKPT else (os.path.basename(os.path.dirname(os.path.dirname(ck))) + "_"
                                            + os.path.splitext(os.path.basename(ck))[0].replace("=", ""))
    for sp in EVAL_SPLITS:
        d = f"{WORK}/eval_{tag}_{sp}"
        if not os.path.exists(f"{d}/summary.json"):
            sh(f"{EVAL_BASE} --split {sp} -p {ck} -o {d}", check=False)
        show_eval(d)
else:
    print("smoke 끝 — 위 숫자를 확인하고 MODE='full' 로 Save & Run All(Commit)")